In [5]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [6]:
raw_path = '../data/raw/tmdb_5000_movies.csv'
df = pd.read_csv(raw_path)

print(f"Total movies loaded: {df.shape[0]:,}")
print(f"Total features:      {df.shape[1]}")

Total movies loaded: 4,803
Total features:      20


In [10]:
df.head(3)

,original_title,budget,genres,homepage,id,keywords,original_language,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,Avatar,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,"In the 22nd century, a paraplegic Marine is di...",150.44,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.00,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.20,11800
1,Pirates of the Caribbean: At World's End,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,"Captain Barbossa, long believed to be dead, ha...",139.08,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.00,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.90,4500
2,Spectre,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,A cryptic message from Bond’s past sends him o...,107.38,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.00,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.30,4466


In [8]:
list(df.columns)

['budget',
 'genres',
 'homepage',
 'id',
 'keywords',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'status',
 'tagline',
 'title',
 'vote_average',
 'vote_count']

In [9]:
df=df[['original_title','budget', 'genres', 'homepage',
        'id', 'keywords', 'original_language', 
         'overview', 'popularity',
        'production_companies', 'production_countries',
        'release_date', 'revenue', 'runtime', 
        'spoken_languages', 'status', 'tagline',
        'title', 'vote_average', 'vote_count']]

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   original_title        4803 non-null   str    
 1   budget                4803 non-null   int64  
 2   genres                4803 non-null   str    
 3   homepage              1712 non-null   str    
 4   id                    4803 non-null   int64  
 5   keywords              4803 non-null   str    
 6   original_language     4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [14]:
zero_budget = (df['budget'] == 0).sum()
zero_revenue = (df['revenue'] == 0).sum()
both_zero = ((df['budget'] == 0) & (df['revenue'] == 0)).sum()

print(f"Total movies:            {len(df)}")
print(f"Zero budget ($0):        {zero_budget} ({zero_budget / len(df) * 100:.1f}%)")
print(f"Zero revenue ($0):       {zero_revenue} ({zero_revenue / len(df) * 100:.1f}%)")
print(f"Both budget & revenue 0: {both_zero} ({both_zero / len(df) * 100:.1f}%)")

Total movies:            4803
Zero budget ($0):        1037 (21.6%)
Zero revenue ($0):       1427 (29.7%)
Both budget & revenue 0: 890 (18.5%)


In [16]:
# Filter out movies with zero budget or revenue
has_valid_finances = (df['budget'] > 0) & (df['revenue'] > 0)
df_clean = df[has_valid_finances].copy()

# Drop rows with missing values in the 'release_date' and 'runtime' columns
df_clean = df_clean.dropna(subset=['release_date', 'runtime'])

# Filter out short movies (with runtime less than 40 minutes)
df_clean = df_clean[df_clean['runtime'] >= 40]

df_clean = df_clean.reset_index(drop=True)

dropped_rows = len(df) - len(df_clean)

print(f"Cleaned data:   {len(df_clean)}")
print(f"Dropped rows:   {dropped_rows}")


Cleaned data:   3229
Dropped rows:   1574


In [17]:
# Helper function to extract names from stringified JSON lists
def parse_json_names(json_str: str) -> list[str]:
 
    if pd.isna(json_str) or not isinstance(json_str, str):
        return []
    try:
        data = json.loads(json_str)
        return [item['name'] for item in data if 'name' in item]
    except (json.JSONDecodeError, TypeError):
        return []

# Extract lists of genre and production company names
df_clean['genres_list'] = df_clean['genres'].apply(parse_json_names)
df_clean['companies_list'] = df_clean['production_companies'].apply(parse_json_names)

# Extract primary genre for single-label categorical grouping
df_clean['primary_genre'] = df_clean['genres_list'].apply(
    lambda genres: genres[0] if len(genres) > 0 else 'Unknown'
)

# Inspect the transformed features alongside movie titles
df_clean[['title', 'primary_genre', 'genres_list']].head(5)

,title,primary_genre,genres_list
0,Avatar,Action,"[Action, Adventure, Fantasy, Science Fiction]"
1,Pirates of the Caribbean: At World's End,Adventure,"[Adventure, Fantasy, Action]"
2,Spectre,Action,"[Action, Adventure, Crime]"
3,The Dark Knight Rises,Action,"[Action, Crime, Drama, Thriller]"
4,John Carter,Action,"[Action, Adventure, Science Fiction]"


In [19]:
# Compute financial profitability metrics
df_clean['net_profit'] = df_clean['revenue'] - df_clean['budget']
df_clean['roi'] = (df_clean['net_profit'] / df_clean['budget']) * 100

# Convert release_date from string to datetime object
df_clean['release_date'] = pd.to_datetime(df_clean['release_date'])

# Extract temporal components for seasonality analysis
df_clean['release_year'] = df_clean['release_date'].dt.year
df_clean['release_month'] = df_clean['release_date'].dt.month
df_clean['release_month_name'] = df_clean['release_date'].dt.month_name()
df_clean['release_dayofweek'] = df_clean['release_date'].dt.day_name()

# Inspect calculated metrics on top releases
display_cols = [
    'title', 'budget', 'revenue', 'net_profit', 
    'roi', 'release_year', 'release_month_name', 'release_dayofweek'
]
df_clean[display_cols].head(5)

,title,budget,revenue,net_profit,roi,release_year,release_month_name,release_dayofweek
0,Avatar,237000000,2787965087,2550965087,"1,076.36",2009,December,Thursday
1,Pirates of the Caribbean: At World's End,300000000,961000000,661000000,220.33,2007,May,Saturday
2,Spectre,245000000,880674609,635674609,259.46,2015,October,Monday
3,The Dark Knight Rises,250000000,1084939099,834939099,333.98,2012,July,Monday
4,John Carter,260000000,284139100,24139100,9.28,2012,March,Wednesday


In [20]:
# Inspect summary statistics for key financial and numerical features
stats_cols = ['budget', 'revenue', 'net_profit', 'roi', 'runtime', 'vote_average']
df_clean[stats_cols].describe()

,budget,revenue,net_profit,roi,runtime,vote_average
count,"3,229.00","3,229.00","3,229.00","3,229.00","3,229.00","3,229.00"
mean,"40,654,444.77","121,242,957.26","80,588,512.49","295,382.24",110.72,6.31
std,"44,396,741.80","186,302,864.01","158,147,947.47","15,061,009.91",20.97,0.87
min,1.00,5.00,"-165,710,090.00",-100.00,41.00,0.00
25%,"10,500,000.00","17,000,000.00","211,394.00",2.25,96.00,5.80
50%,"25,000,000.00","55,184,721.00","26,165,571.00",130.04,107.00,6.30
75%,"55,000,000.00","146,292,009.00","97,489,822.00",342.08,121.00,6.90
max,"380,000,000.00","2,787,965,087.00","2,550,965,087.00","849,999,900.00",338.00,8.50


In [25]:
df[df["budget"] < 100000].sort_values(by="budget", ascending=False)[["title", "budget", "revenue"]].head(75)

,title,budget,revenue
3159,The Texas Chain Saw Massacre,85000,30859000
4741,Queen Crab,75000,0
4748,Absentia,70000,0
4742,Super Size Me,65000,28575078
4752,Tiny Furniture,65000,416498
...,...,...,...
4181,Ed and His Dead Mother,0,0
4180,The Greatest Movie Ever Sold,0,621951
4155,The Legend of Hell's Gate: An American Conspiracy,0,0
4340,Chernobyl Diaries,0,18112929


In [26]:
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   original_title        4803 non-null   str    
 1   budget                4803 non-null   int64  
 2   genres                4803 non-null   str    
 3   homepage              1712 non-null   str    
 4   id                    4803 non-null   int64  
 5   keywords              4803 non-null   str    
 6   original_language     4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

original_title             0
budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
dtype: int64

## Data Quality Audit & Cleaning Summary

### Identified Dataset Issues
1. **Zero-value Financial Anomalies:**
    - **Issue:** 21.6% of movies had a `$0` budget and 29.7% had `$0` revenue (placeholder for missing metadata).
    - **Impact:** Distorts average industry metrics and causes division-by-zero errors when computing ROI.
    - **Resolution:** Retained only records where both `budget > 0` and `revenue > 0` (3,229 movies retained).

2. **Serialized JSON Metadata:**
    - **Issue:** Columns `genres` and `production_companies` were stored as stringified JSON arrays.
    - **Resolution:** Extracted names into native Python lists and isolated a single `primary_genre` for categorical grouping.

3. **Missing Critical Values & Outliers:**
    - **Issue:** Missing values in `release_date` and `runtime`, alongside short promo videos (< 40 min).
    - **Resolution:** Dropped records with null metadata and enforced a standard feature film threshold (`runtime >= 40`).

4. **Extreme ROI Skew:**
    - **Finding:** A record with an input budget of `$1.00` inflated the maximum ROI to ~850,000,000%.
    - **Note for EDA:** Future hypothesis testing will rely on **median** figures rather than arithmetic means to avoid distortion from extreme outliers.

In [27]:
# Select key columns and export cleaned dataset
final_columns = [
    'id', 'title', 'budget', 'revenue', 'net_profit', 'roi',
    'vote_average', 'vote_count', 'popularity', 'runtime',
    'primary_genre', 'genres_list', 'companies_list',
    'release_date', 'release_year', 'release_month', 
    'release_month_name', 'release_dayofweek'
]
df_final = df_clean[final_columns].copy()

output_path = '../data/processed/tmdb_movies_cleaned.csv'
df_final.to_csv(output_path, index=False)
print(f"Dataset successfully saved to: {output_path}")

Dataset successfully saved to: ../data/processed/tmdb_movies_cleaned.csv


In [28]:
# Define range cutoffs
budget_bins = [0, 10_000_000, 50_000_000, np.inf]

# Define readable business labels corresponding to intervals
budget_labels = [
    'Low Budget (<$10M)', 
    'Mid Budget ($10M-$50M)', 
    'Blockbuster (>$50M)'
]

# Bin continuous budget values into categorical tiers
df_clean['budget_tier'] = pd.cut(
    df_clean['budget'],
    bins=budget_bins,
    labels=budget_labels,
    right=False
)

# Inspect category counts
df_clean['budget_tier'].value_counts()

budget_tier
Mid Budget ($10M-$50M)    1541
Blockbuster (>$50M)        956
Low Budget (<$10M)         732
Name: count, dtype: int64

In [30]:
import numpy as np

df_clean['is_profitable'] = df_clean['revenue'] > df_clean['budget']
df_clean['roi_multiplier'] = (df_clean['revenue'] - df_clean['budget']) / df_clean['budget']

budget_bins = [0, 10_000_000, 50_000_000, np.inf]
budget_labels = ['Low Budget (<$10M)', 'Mid Budget ($10M-$50M)', 'Blockbuster (>$50M)']
df_clean['budget_tier'] = pd.cut(df_clean['budget'], bins=budget_bins, labels=budget_labels, right=False)

In [31]:
final_columns = [
    'id', 'title', 'budget', 'revenue', 'net_profit', 
    'roi', 'roi_multiplier', 'is_profitable', 'budget_tier',
    'vote_average', 'vote_count', 'popularity', 'runtime',
    'primary_genre', 'genres_list', 'companies_list',
    'release_date', 'release_year', 'release_month', 
    'release_month_name', 'release_dayofweek'
]

df_final = df_clean[final_columns].copy()
output_path = '../data/processed/tmdb_movies_cleaned.csv'
df_final.to_csv(output_path, index=False)

print(f"Cleaned dataset updated: {df_final.shape[0]} rows, {df_final.shape[1]} cols")

Cleaned dataset updated: 3229 rows, 21 cols


In [33]:
df_final.head(3)

,id,title,budget,revenue,net_profit,roi,roi_multiplier,is_profitable,budget_tier,vote_average,vote_count,popularity,runtime,primary_genre,genres_list,companies_list,release_date,release_year,release_month,release_month_name,release_dayofweek
0,19995,Avatar,237000000,2787965087,2550965087,"1,076.36",10.76,True,Blockbuster (>$50M),7.20,11800,150.44,162.00,Action,"[Action, Adventure, Fantasy, Science Fiction]","[Ingenious Film Partners, Twentieth Century Fo...",2009-12-10,2009,12,December,Thursday
1,285,Pirates of the Caribbean: At World's End,300000000,961000000,661000000,220.33,2.20,True,Blockbuster (>$50M),6.90,4500,139.08,169.00,Adventure,"[Adventure, Fantasy, Action]","[Walt Disney Pictures, Jerry Bruckheimer Films...",2007-05-19,2007,5,May,Saturday
2,206647,Spectre,245000000,880674609,635674609,259.46,2.59,True,Blockbuster (>$50M),6.30,4466,107.38,148.00,Action,"[Action, Adventure, Crime]","[Columbia Pictures, Danjaq, B24]",2015-10-26,2015,10,October,Monday
